# Características RoCoLe — inventario HSV con máscara COCO

Objetivo: armar un inventario pequeño de histogramas HSV solo dentro de la máscara Leaf de COCO, con el mismo índice y split ya publicados, y guardarlo en un CSV paralelo sin entrenar nada. Es una rama de desarrollo: mismas fotos y partición que el masked RGB/GLCM, pero midiendo solo color en HSV dentro de la hoja. No reescribe el 02 actual.


## Llegar al índice, al COCO y fijar la semilla

Usamos la carpeta featured, el índice partido, el JSON oficial de anotaciones y una carpeta de figuras. Semilla 42.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from pycocotools.coco import COCO
from skimage.color import rgb2hsv
from tqdm.auto import tqdm


D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# semilla fija para muestreos reproducibles
seed = 42
repo_root = Path("../..").resolve()
featured_dir = repo_root / "data" / "featured" / "rocole"
index_path = featured_dir / "index.csv"
coco_json_path = (
    repo_root / "data" / "raw" / "RoCoLe" / "Annotations" / "RoCoLE-coco.json"
)
results_dir = repo_root / "experiments" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

print("repo_root:", repo_root)
print("index_path:", index_path)
print("existe índice:", index_path.exists())
print("coco_json_path:", coco_json_path)
print("existe coco:", coco_json_path.exists())
print("results_dir:", results_dir)
seed


repo_root: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe
index_path: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\data\featured\rocole\index.csv
existe índice: True
coco_json_path: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\data\raw\RoCoLe\Annotations\RoCoLE-coco.json
existe coco: True
results_dir: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results


42

## Cargar el índice partido

El índice es la mesa de trabajo: una fila por foto, con etiqueta, partición y ruta. No rehacemos el split.


In [3]:
rocole_index = pd.read_csv(index_path)

print("filas:", len(rocole_index))
print("columnas:", list(rocole_index.columns))
rocole_index.head()


filas: 1481
columnas: ['File', 'plant_key', 'link_group', 'Binary.Label', 'Multiclass.Label', 'leaf_face', 'split', 'photo_path']


,File,plant_key,link_group,Binary.Label,Multiclass.Label,leaf_face,split,photo_path
0,C1P1H1.jpg,C1P1,74,healthy,healthy,H,train,data/curated/rocole/photos/C1P1H1.jpg
1,C1P2E2.jpg,C1P2,84,unhealthy,rust_level_2,E,train,data/curated/rocole/photos/C1P2E2.jpg
2,C1P2H1.jpg,C1P2,84,healthy,healthy,H,train,data/curated/rocole/photos/C1P2H1.jpg
3,C1P3E1.jpg,C1P3,95,healthy,healthy,E,train,data/curated/rocole/photos/C1P3E1.jpg
4,C1P3E2.jpg,C1P3,95,unhealthy,rust_level_2,E,train,data/curated/rocole/photos/C1P3E2.jpg


## Columnas de identidad que viajan con cada fila

Cada fila debe llevar su identidad: archivo, partición, grupo de enlace y las dos etiquetas. Esas columnas viajan enteras hasta el CSV final.


In [4]:
# rocole_index: mesa con File, split, link_group y etiquetas

print("filas:", len(rocole_index))
print("splits:", rocole_index["split"].value_counts().to_dict())
id_columns = [
    "File",
    "split",
    "link_group",
    "Binary.Label",
    "Multiclass.Label",
]

print("id_columns:", id_columns)
rocole_index[id_columns].head()


filas: 1481
splits: {'train': 1037, 'test': 226, 'val': 218}
id_columns: ['File', 'split', 'link_group', 'Binary.Label', 'Multiclass.Label']


,File,split,link_group,Binary.Label,Multiclass.Label
0,C1P1H1.jpg,train,74,healthy,healthy
1,C1P2E2.jpg,train,84,unhealthy,rust_level_2
2,C1P2H1.jpg,train,84,healthy,healthy
3,C1P3E1.jpg,train,95,healthy,healthy
4,C1P3E2.jpg,train,95,unhealthy,rust_level_2


## Cargar las anotaciones COCO

Las máscaras oficiales viven en el JSON COCO de RoCoLe. Lo cargamos una vez para pedir la máscara Leaf por nombre de archivo.


In [5]:
# si falta: pip install pycocotools

coco = COCO(str(coco_json_path))

print("coco_json_path:", coco_json_path)
print("imágenes en COCO:", len(coco.getImgIds()))
coco


loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
coco_json_path: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\data\raw\RoCoLe\Annotations\RoCoLE-coco.json
imágenes en COCO: 1560


## Tamaños del histograma HSV

El histograma será chico a propósito: ocho cajones de matiz, cuatro de saturación y cuatro de valor. Eso da dieciséis columnas con nombres fijos.


In [6]:
hue_bins = 8
saturation_bins = 4
value_bins = 4
feature_columns = (
    [f"hsv_h_b{i}" for i in range(hue_bins)]
    + [f"hsv_s_b{i}" for i in range(saturation_bins)]
    + [f"hsv_v_b{i}" for i in range(value_bins)]
)

print("hue_bins:", hue_bins)
print("saturation_bins:", saturation_bins)
print("value_bins:", value_bins)
print("n_features:", len(feature_columns))
feature_columns


hue_bins: 8
saturation_bins: 4
value_bins: 4
n_features: 16


['hsv_h_b0',
 'hsv_h_b1',
 'hsv_h_b2',
 'hsv_h_b3',
 'hsv_h_b4',
 'hsv_h_b5',
 'hsv_h_b6',
 'hsv_h_b7',
 'hsv_s_b0',
 'hsv_s_b1',
 'hsv_s_b2',
 'hsv_s_b3',
 'hsv_v_b0',
 'hsv_v_b1',
 'hsv_v_b2',
 'hsv_v_b3']

## Cargar la máscara COCO por File

En el JSON, file_name es una URL; emparejamos si termina en el File del índice. Devolvemos la unión (OR) de las anotaciones Leaf. Si no hay máscara o no calza, el fallo queda explícito: no caemos a foto completa.


In [7]:
# coco: anotaciones RoCoLe ya cargadas

def load_coco_leaf_mask(coco, file_name, height, width):
    """Máscara Leaf (OR) emparejando file_name del JSON por sufijo File."""
    matched_images = [
        img
        for img in coco.loadImgs(coco.getImgIds())
        if img["file_name"].endswith(file_name)
    ]
    if len(matched_images) != 1:
        raise ValueError(
            f"esperaba 1 imagen COCO para {file_name}, hay {len(matched_images)}"
        )

    coco_image = matched_images[0]
    # categoría Leaf = 1; imgIds en lista (ids Labelbox son strings)
    ann_ids = coco.getAnnIds(imgIds=[coco_image["id"]], catIds=[1])
    if len(ann_ids) < 1:
        # respaldo por nombre de categoría si catIds=[1] no encuentra
        leaf_cat_ids = [
            cat["id"]
            for cat in coco.loadCats(coco.getCatIds())
            if cat.get("name", "").lower() == "leaf"
        ]
        ann_ids = coco.getAnnIds(imgIds=[coco_image["id"]], catIds=leaf_cat_ids)

    annotations = coco.loadAnns(ann_ids)
    if len(annotations) < 1:
        raise ValueError(f"sin anotaciones Leaf para {file_name}")

    leaf_mask = None
    for annotation in annotations:
        mask_i = coco.annToMask(annotation).astype(bool)
        leaf_mask = mask_i if leaf_mask is None else (leaf_mask | mask_i)

    if leaf_mask.shape != (height, width):
        raise ValueError(
            f"máscara {leaf_mask.shape} vs imagen {(height, width)} para {file_name}"
        )
    return leaf_mask


load_coco_leaf_mask


<function __main__.load_coco_leaf_mask(coco, file_name, height, width)>

## Convertir RGB a HSV

Pasamos la foto de RGB a HSV. Esa conversión es solo el cambio de espacio de color; todavía no armamos el histograma.


In [8]:
def rgb_to_hsv(image):
    """RGB uint8 o float → HSV float con H,S,V en [0, 1]."""
    rgb = np.asarray(image, dtype=np.float64)
    if rgb.max() > 1.0:
        rgb = rgb / 255.0
    hsv = rgb2hsv(rgb)
    return hsv


rgb_to_hsv


<function __main__.rgb_to_hsv(image)>

## Histograma HSV solo dentro de la máscara

Sobre los píxeles de la máscara contamos el histograma de cada canal, lo normalizamos para que sume uno dentro de la hoja, y devolvemos las dieciséis columnas. Máscara vacía = error explícito.


In [9]:
# rgb_to_hsv, hue_bins, saturation_bins, value_bins, feature_columns

def extract_hsv_hist_masked(image, leaf_mask):
    """Histograma H/S/V normalizado (suma 1 por canal) solo en la máscara."""
    if not np.any(leaf_mask):
        raise ValueError("máscara Leaf vacía")

    hsv = rgb_to_hsv(image)
    hue_values = hsv[:, :, 0][leaf_mask]
    saturation_values = hsv[:, :, 1][leaf_mask]
    value_values = hsv[:, :, 2][leaf_mask]

    hue_hist, _ = np.histogram(hue_values, bins=hue_bins, range=(0.0, 1.0))
    saturation_hist, _ = np.histogram(
        saturation_values, bins=saturation_bins, range=(0.0, 1.0)
    )
    value_hist, _ = np.histogram(
        value_values, bins=value_bins, range=(0.0, 1.0)
    )

    hue_hist = hue_hist.astype(np.float64)
    saturation_hist = saturation_hist.astype(np.float64)
    value_hist = value_hist.astype(np.float64)

    hue_sum = hue_hist.sum()
    saturation_sum = saturation_hist.sum()
    value_sum = value_hist.sum()
    if hue_sum <= 0 or saturation_sum <= 0 or value_sum <= 0:
        raise ValueError("histograma vacío dentro de la máscara")

    hue_hist = hue_hist / hue_sum
    saturation_hist = saturation_hist / saturation_sum
    value_hist = value_hist / value_sum

    values = np.concatenate([hue_hist, saturation_hist, value_hist])
    return {
        column_name: float(values[i])
        for i, column_name in enumerate(feature_columns)
    }


extract_hsv_hist_masked


<function __main__.extract_hsv_hist_masked(image, leaf_mask)>

## Filas de smoke (train)

Antes del barrido largo tomamos unas pocas filas de train, mezclando sanas y afectadas, con la semilla fija. Solo para probar que el pipeline no se cae.


In [10]:
# rocole_index, seed, id_columns

train_rows = rocole_index.loc[rocole_index["split"] == "train"].reset_index(drop=True)
healthy_train_sample = (
    train_rows.loc[train_rows["Binary.Label"] == "healthy"]
    .sample(n=3, random_state=seed)
    .reset_index(drop=True)
)
unhealthy_train_sample = (
    train_rows.loc[train_rows["Binary.Label"] == "unhealthy"]
    .sample(n=3, random_state=seed)
    .reset_index(drop=True)
)
smoke_train_rows = pd.concat(
    [healthy_train_sample, unhealthy_train_sample],
    ignore_index=True,
)

print("smoke_train_rows:", len(smoke_train_rows))
print(smoke_train_rows["Binary.Label"].value_counts().to_string())
smoke_train_rows[id_columns]


smoke_train_rows: 6
Binary.Label
healthy      3
unhealthy    3


,File,split,link_group,Binary.Label,Multiclass.Label
0,C4P39H1.jpg,train,190,healthy,healthy
1,C9P20E2.jpg,train,333,healthy,healthy
2,C1P6H1.jpg,train,105,healthy,healthy
3,C1P26H2.jpg,train,91,unhealthy,rust_level_2
4,C6P40E2.jpg,train,254,unhealthy,red_spider_mite
5,C12P22H2.jpg,train,66,unhealthy,rust_level_1


## Smoke: histograma HSV con máscara, fila a fila

Sobre esas pocas filas leemos la foto, pedimos la máscara Leaf y medimos el histograma HSV. Si falla, NaN en features y mensaje en error. Sin fallback a imagen completa.


In [11]:
# smoke_train_rows, coco, repo_root
# load_coco_leaf_mask, extract_hsv_hist_masked
# feature_columns, id_columns

smoke_feature_rows = []
for _, row in smoke_train_rows.iterrows():
    feature_row = {column_name: row[column_name] for column_name in id_columns}
    feature_row["error"] = None
    for column_name in feature_columns:
        feature_row[column_name] = np.nan

    try:
        photo_path = repo_root / row["photo_path"]
        sample_image = Image.open(photo_path).convert("RGB")
        height, width = sample_image.size[1], sample_image.size[0]
        leaf_mask = load_coco_leaf_mask(coco, row["File"], height, width)
        hsv_summary = extract_hsv_hist_masked(sample_image, leaf_mask)
        feature_row.update(hsv_summary)
    except Exception as exc:
        # fallo de máscara o lectura: NaN en features, mensaje en error
        feature_row["error"] = str(exc)

    smoke_feature_rows.append(feature_row)

print("smoke_feature_rows:", len(smoke_feature_rows))
print("con error:", sum(1 for r in smoke_feature_rows if r["error"]))
smoke_feature_rows


smoke_feature_rows: 6
con error: 0


[{'File': 'C4P39H1.jpg',
  'split': 'train',
  'link_group': 190,
  'Binary.Label': 'healthy',
  'Multiclass.Label': 'healthy',
  'error': None,
  'hsv_h_b0': 0.0006259016838774334,
  'hsv_h_b1': 0.6923252706428249,
  'hsv_h_b2': 0.30703965022925256,
  'hsv_h_b3': 2.753233213537097e-06,
  'hsv_h_b4': 1.8354888090247315e-06,
  'hsv_h_b5': 2.753233213537097e-06,
  'hsv_h_b6': 9.177444045123657e-07,
  'hsv_h_b7': 9.177444045123657e-07,
  'hsv_s_b0': 0.003543411145822244,
  'hsv_s_b1': 0.01736005315575591,
  'hsv_s_b2': 0.05475905538403932,
  'hsv_s_b3': 0.9243374803143826,
  'hsv_v_b0': 0.00017987790328442368,
  'hsv_v_b1': 0.6115793647006135,
  'hsv_v_b2': 0.38372820815911485,
  'hsv_v_b3': 0.004512549236987302},
 {'File': 'C9P20E2.jpg',
  'split': 'train',
  'link_group': 333,
  'Binary.Label': 'healthy',
  'Multiclass.Label': 'healthy',
  'error': None,
  'hsv_h_b0': 0.0037835877744576904,
  'hsv_h_b1': 0.3653523430977633,
  'hsv_h_b2': 0.6301922433639637,
  'hsv_h_b3': 0.0005059775627

## Tabla smoke y chequeo de columnas / sumas

Pasamos el smoke a tabla y comprobamos que estén las dieciséis columnas pedidas y que, sin error, los números sean finitos y sumen cerca de uno por canal.


In [12]:
# smoke_feature_rows, feature_columns, id_columns
# hue_bins, saturation_bins, value_bins

smoke_feature_table = pd.DataFrame(smoke_feature_rows)

expected_columns = id_columns + feature_columns + ["error"]
missing_columns = [
    name for name in expected_columns if name not in smoke_feature_table.columns
]
assert missing_columns == [], f"faltan columnas: {missing_columns}"

ok_mask = smoke_feature_table["error"].isna()
finite_ok = True
channel_sums_ok = True
channel_sum_report = []
if ok_mask.any():
    feature_block = smoke_feature_table.loc[ok_mask, feature_columns]
    finite_ok = bool(np.isfinite(feature_block.to_numpy(dtype=float)).all())

    hue_cols = feature_columns[:hue_bins]
    sat_cols = feature_columns[hue_bins : hue_bins + saturation_bins]
    val_cols = feature_columns[hue_bins + saturation_bins :]
    for idx, row in feature_block.iterrows():
        hue_sum = float(row[hue_cols].sum())
        sat_sum = float(row[sat_cols].sum())
        val_sum = float(row[val_cols].sum())
        channel_sum_report.append(
            {
                "row": int(idx),
                "hue_sum": hue_sum,
                "sat_sum": sat_sum,
                "val_sum": val_sum,
            }
        )
        if not (
            np.isclose(hue_sum, 1.0, atol=1e-6)
            and np.isclose(sat_sum, 1.0, atol=1e-6)
            and np.isclose(val_sum, 1.0, atol=1e-6)
        ):
            channel_sums_ok = False

print("filas:", len(smoke_feature_table))
print("columnas esperadas ok:", missing_columns == [])
print("finitos en filas sin error:", finite_ok)
print("sumas por canal ~1:", channel_sums_ok)
print(channel_sum_report)
smoke_feature_table


filas: 6
columnas esperadas ok: True
finitos en filas sin error: True
sumas por canal ~1: True
[{'row': 0, 'hue_sum': 1.0, 'sat_sum': 1.0, 'val_sum': 1.0}, {'row': 1, 'hue_sum': 1.0000000000000002, 'sat_sum': 0.9999999999999999, 'val_sum': 1.0}, {'row': 2, 'hue_sum': 0.9999999999999999, 'sat_sum': 1.0, 'val_sum': 1.0}, {'row': 3, 'hue_sum': 1.0, 'sat_sum': 1.0, 'val_sum': 1.0}, {'row': 4, 'hue_sum': 1.0, 'sat_sum': 0.9999999999999999, 'val_sum': 1.0}, {'row': 5, 'hue_sum': 1.0, 'sat_sum': 1.0, 'val_sum': 1.0}]


,File,split,link_group,Binary.Label,Multiclass.Label,error,hsv_h_b0,hsv_h_b1,hsv_h_b2,hsv_h_b3,...,hsv_h_b6,hsv_h_b7,hsv_s_b0,hsv_s_b1,hsv_s_b2,hsv_s_b3,hsv_v_b0,hsv_v_b1,hsv_v_b2,hsv_v_b3
0,C4P39H1.jpg,train,190,healthy,healthy,None,0.000626,0.692325,0.307040,0.000003,...,9.177444e-07,9.177444e-07,0.003543,0.017360,0.054759,0.924337,0.000180,0.611579,0.383728,0.004513
1,C9P20E2.jpg,train,333,healthy,healthy,None,0.003784,0.365352,0.630192,0.000506,...,3.654282e-05,1.293054e-04,0.651609,0.344481,0.003904,0.000006,0.005768,0.120940,0.575181,0.298111
2,C1P6H1.jpg,train,105,healthy,healthy,None,0.000426,0.602718,0.396396,0.000389,...,3.404883e-05,3.688623e-05,0.033257,0.231487,0.375681,0.359576,0.001368,0.006061,0.733003,0.259568
3,C1P26H2.jpg,train,91,unhealthy,rust_level_2,None,0.013975,0.174808,0.799393,0.000967,...,2.915274e-03,7.104384e-03,0.015589,0.457258,0.494696,0.032457,0.005661,0.052396,0.807104,0.134838
4,C6P40E2.jpg,train,254,unhealthy,red_spider_mite,None,0.016977,0.125257,0.769190,0.082726,...,0.000000e+00,5.850618e-03,0.706034,0.291840,0.002123,0.000004,0.000557,0.025055,0.876548,0.097841
5,C12P22H2.jpg,train,66,unhealthy,rust_level_1,None,0.004362,0.062742,0.930694,0.000001,...,0.000000e+00,2.200679e-03,0.003699,0.196708,0.288377,0.511216,0.000003,0.002674,0.910991,0.086332


## Barrido completo del índice (con máscara)

Recorremos todas las filas del índice con el mismo protocolo masked. Si no hay máscara Leaf, queda error y NaN. No caemos a foto completa.


In [13]:
# rocole_index, coco, repo_root
# load_coco_leaf_mask, extract_hsv_hist_masked
# feature_columns, id_columns

inventory_feature_rows = []
for _, row in tqdm(
    rocole_index.iterrows(),
    total=len(rocole_index),
    desc="inventario HSV masked",
):
    feature_row = {column_name: row[column_name] for column_name in id_columns}
    feature_row["error"] = None
    for column_name in feature_columns:
        feature_row[column_name] = np.nan

    try:
        photo_path = repo_root / row["photo_path"]
        sample_image = Image.open(photo_path).convert("RGB")
        height, width = sample_image.size[1], sample_image.size[0]
        # sin fallback a full-image: si no hay máscara, Exception → error
        leaf_mask = load_coco_leaf_mask(coco, row["File"], height, width)
        hsv_summary = extract_hsv_hist_masked(sample_image, leaf_mask)
        feature_row.update(hsv_summary)
    except Exception as exc:
        feature_row["error"] = str(exc)

    inventory_feature_rows.append(feature_row)

print("inventory_feature_rows:", len(inventory_feature_rows))
print(
    "con error:",
    sum(1 for r in inventory_feature_rows if r["error"] is not None),
)


inventario HSV masked:   0%|          | 0/1481 [00:00<?, ?it/s]

inventario HSV masked:   0%|          | 1/1481 [00:00<12:47,  1.93it/s]

inventario HSV masked:   0%|          | 2/1481 [00:01<12:30,  1.97it/s]

inventario HSV masked:   0%|          | 3/1481 [00:01<12:28,  1.97it/s]

inventario HSV masked:   0%|          | 4/1481 [00:02<12:22,  1.99it/s]

inventario HSV masked:   0%|          | 5/1481 [00:02<12:10,  2.02it/s]

... (1,474 outputs left out)

inventario HSV masked: 100%|█████████▉| 1480/1481 [17:49<00:00,  2.33it/s]

inventario HSV masked: 100%|██████████| 1481/1481 [17:50<00:00,  2.18it/s]

inventario HSV masked: 100%|██████████| 1481/1481 [17:50<00:00,  1.38it/s]

inventory_feature_rows: 1481
con error: 0


## Mesa de features del inventario HSV masked

La lista del barrido se convierte en la mesa hsv_hist_features_masked: identidad, dieciséis bins y error.


In [14]:
# inventory_feature_rows

hsv_hist_features_masked = pd.DataFrame(inventory_feature_rows)

print("filas:", len(hsv_hist_features_masked))
print("columnas:", list(hsv_hist_features_masked.columns))
hsv_hist_features_masked.head()


filas: 1481
columnas: ['File', 'split', 'link_group', 'Binary.Label', 'Multiclass.Label', 'error', 'hsv_h_b0', 'hsv_h_b1', 'hsv_h_b2', 'hsv_h_b3', 'hsv_h_b4', 'hsv_h_b5', 'hsv_h_b6', 'hsv_h_b7', 'hsv_s_b0', 'hsv_s_b1', 'hsv_s_b2', 'hsv_s_b3', 'hsv_v_b0', 'hsv_v_b1', 'hsv_v_b2', 'hsv_v_b3']


,File,split,link_group,Binary.Label,Multiclass.Label,error,hsv_h_b0,hsv_h_b1,hsv_h_b2,hsv_h_b3,...,hsv_h_b6,hsv_h_b7,hsv_s_b0,hsv_s_b1,hsv_s_b2,hsv_s_b3,hsv_v_b0,hsv_v_b1,hsv_v_b2,hsv_v_b3
0,C1P1H1.jpg,train,74,healthy,healthy,None,0.000180,0.011176,0.944385,0.044259,...,0.000000,0.000000,0.010138,0.429587,0.452823,0.107453,0.000353,0.305308,0.662189,0.032151
1,C1P2E2.jpg,train,84,unhealthy,rust_level_2,None,0.039306,0.355705,0.604945,0.000010,...,0.000002,0.000027,0.004484,0.442161,0.552731,0.000624,0.000522,0.015845,0.827815,0.155819
2,C1P2H1.jpg,train,84,healthy,healthy,None,0.000208,0.127795,0.818221,0.053776,...,0.000000,0.000000,0.129314,0.303065,0.368171,0.199449,0.000242,0.031814,0.624172,0.343772
3,C1P3E1.jpg,train,95,healthy,healthy,None,0.001579,0.313902,0.684165,0.000024,...,0.000036,0.000282,0.106949,0.791879,0.100388,0.000784,0.001639,0.012354,0.772775,0.213232
4,C1P3E2.jpg,train,95,unhealthy,rust_level_2,None,0.081139,0.852689,0.062831,0.000038,...,0.000016,0.003287,0.006619,0.695575,0.296780,0.001025,0.001270,0.030871,0.600927,0.366932


## Chequeo de filas del inventario

Comprobamos una fila por foto del índice, que los conteos por split cuadren y cuántas filas trajeron error.


In [15]:
# hsv_hist_features_masked, rocole_index

inventory_check = {
    "n_index": len(rocole_index),
    "n_features": len(hsv_hist_features_masked),
    "len_match": len(hsv_hist_features_masked) == len(rocole_index),
    "split_counts_index": rocole_index["split"].value_counts().to_dict(),
    "split_counts_features": hsv_hist_features_masked["split"]
    .value_counts()
    .to_dict(),
    "n_errors": int(hsv_hist_features_masked["error"].notna().sum()),
}

assert inventory_check["len_match"], (
    f"filas features {inventory_check['n_features']} "
    f"!= índice {inventory_check['n_index']}"
)

inventory_check


{'n_index': 1481,
 'n_features': 1481,
 'len_match': True,
 'split_counts_index': {'train': 1037, 'test': 226, 'val': 218},
 'split_counts_features': {'train': 1037, 'test': 226, 'val': 218},
 'n_errors': 0}

## Guardar el CSV de features HSV masked

Escribimos hsv_hist_features_masked.csv en la carpeta featured de RoCoLe. Aquí termina la extracción; el EDA va en el otro cuaderno.


In [16]:
# hsv_hist_features_masked, featured_dir

features_csv_path = featured_dir / "hsv_hist_features_masked.csv"
hsv_hist_features_masked.to_csv(features_csv_path, index=False)

print("guardado:", features_csv_path)
print("existe:", features_csv_path.exists())
features_csv_path


guardado: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\data\featured\rocole\hsv_hist_features_masked.csv
existe: True


WindowsPath('D:/Maestria/Documentos/Proyecto de innovación tecnologica en IA/Desarrollo/diagnostico-hojas-cafe/data/featured/rocole/hsv_hist_features_masked.csv')

## Conclusiones

Quedó un CSV paralelo de dieciséis bins HSV solo dentro de la máscara Leaf; mismas filas y split que el índice; sin foto completa y sin modelo.
